# ДЗ 11. Интегрирование рациональных функций

## Теоретические задания

### 1) Что такое простейшая дробь в $\mathbb{Q}$?

Простейшая дробь в $\mathbb{Q}$ — это правильная дробь $\frac{c}{p^m}$, где $p$ — простое число, $m \in \mathbb{N}$, и $0 \le c < p$.

Например, $\frac{1}{8} = \frac{1}{2^3}$ — простейшая дробь, поскольку знаменатель $8 = 2^3$ есть степень простого числа $2$, а числитель $1 < 2$.

Всякое рациональное число можно представить как сумму целого числа и нескольких простейших дробей (теорема 24 из лекции).

### 2) Что такое простейшая дробь в поле частных кольца $\mathbb{Q}[x]$?

Простейшая дробь в поле частных кольца $k[x]$ — это правильная дробь $\frac{q}{p^m}$, где $p$ — неприводимый (простой) многочлен из $k[x]$, $m \in \mathbb{N}$, а $\deg q < \deg p$.

Над $\mathbb{Q}$ простые многочлены — это многочлены первой степени $x - a$ и неприводимые квадратные трёхчлены $x^2 + a_1 x + a_0$ с отрицательным дискриминантом. Соответственно простейшие дроби бывают двух типов:
$$\frac{b}{(x-a)^m}, \qquad \frac{b_1 x + b_0}{(x^2 + a_1 x + a_0)^m}.$$

### 3) Как разложить дробь на простейшие в поле частных кольца $\mathbb{Q}[x]$?

Алгоритм аналогичен разложению рациональных чисел:

1. Если дробь неправильная, выделяем целую (полиномиальную) часть делением с остатком: $f/g = q + r/g$, где $\deg r < \deg g$.
2. Раскладываем знаменатель $g$ на неприводимые множители: $g = p_1^{m_1} \cdots p_s^{m_s}$.
3. Последовательно находим коэффициенты $c_i$ простейших дробей, решая линейные сравнения по модулю $p_i$ (как в доказательстве теоремы 24, но вместо $\text{GF}(p)$ используем $k[x]/(p_i)$). На каждом шаге степень $p_i$ в знаменателе уменьшается на единицу.

В Sage это делает метод `partial_fraction_decomposition()` у элементов `FractionField(k[x])`.

### 4) Как вычислить интеграл $\int \frac{b_0 + b_1 x}{a_0 + a_1 x + x^2}\,dx$ при условии, что знаменатель не имеет вещественных нулей?

Поскольку дискриминант $a_1^2 - 4a_0 < 0$, обозначим $s = \sqrt{4a_0 - a_1^2}$. Выделим полный квадрат в знаменателе:
$$x^2 + a_1 x + a_0 = \left(x + \frac{a_1}{2}\right)^2 + \frac{4a_0 - a_1^2}{4}.$$

Представляя числитель как $b_1\left(x + \frac{a_1}{2}\right) + \left(b_0 - \frac{a_1 b_1}{2}\right)$, получаем:

$$\int \frac{b_0 + b_1 x}{x^2 + a_1 x + a_0}\,dx = \frac{b_1}{2}\ln(x^2 + a_1 x + a_0) + \frac{2b_0 - a_1 b_1}{s}\arctan\frac{2x + a_1}{s}.$$

Под логарифмом стоит всегда положительное выражение (поскольку нет вещественных корней), а $s > 0$.

---
## Практические задания

In [ ]:
var('x')

### Задача 1. Разложить дробь $14515/39168$ на простые дроби.

In [ ]:
QQ(14515/39168).partial_fraction_decomposition()

Проверим вручную по алгоритму из лекции:

In [ ]:
ZZ(39168).factor()

In [ ]:
# 39168 = 2^6 * 3 * 2^2 * ... пусть factor покажет
# Разложим по алгоритму из лекции: последовательно снимаем простые
r = QQ(14515)/QQ(39168)
pfd = QQ(r).partial_fraction_decomposition()
print('Разложение:', pfd)
print('Проверка:', pfd[0] + sum(pfd[1]))

### Задача 2. Разложить дробь $\frac{x^5}{x^5 + 3x + 1}$ на простые дроби в поле частных кольца $\mathbb{A}[x]$.

In [ ]:
f = x^5/(x^5 + 3*x + 1)
PFD = FractionField(AA[x])(f).partial_fraction_decomposition()
print('Полиномиальная часть:', PFD[0])
print('Простейшие дроби:')
for fr in PFD[1]:
    print(' ', fr)

### Задача 3. Функция для вычисления интеграла от многочлена (без `integral`).

In [ ]:
def poly_integral(p, x):
    """Интеграл многочлена p по переменной x."""
    p = QQ[x](p)
    return sum(p[k]*x^(k+1)/(k+1) for k in range(p.degree()+1))

In [ ]:
# Тесты
print(poly_integral(3*x^2 + 2*x + 1, x))  # x^3 + x^2 + x
print(poly_integral(x^5, x))                # x^6/6
print(poly_integral(1, x))                  # x

### Задача 4. Функция для вычисления интеграла от рациональной функции над полем $\mathbb{A}$.

Используем `partial_fraction_decomposition` и `pfdintegral` из лекций.

In [ ]:
def pfdintegral(f, x):
    """Интеграл простейшей дроби из поля частных AA[x]. Из лекции."""
    g = (f).numerator()
    h = (f).denominator()
    if AA[x](h).degree() == 1:
        return g*ln(abs(x + h.subs(x=0)))
    else:
        b0 = g.subs(x=0)
        b1 = diff(g, x)
        a0 = h.subs(x=0)
        a1 = diff(h, x).subs(x=0)
        s = sqrt(-a1^2 + 4*a0)
        return 1/2*b1*log(a1*x + x^2 + a0) - (a1*b1 - 2*b0)*arctan((2*x + a1)/s)/s

def rat_integral(f, x):
    """Интеграл рациональной функции f(x) над полем A."""
    PFD = FractionField(AA[x])(f).partial_fraction_decomposition()
    # Полиномиальная часть
    F = poly_integral(PFD[0], x)
    # Простейшие дроби
    F += sum(pfdintegral(fr, x) for fr in PFD[1])
    return F

In [ ]:
# Тест: интеграл из примера 38 лекции
f = x^5/((x+1)*(x-2)*(x-3)^2)
F = rat_integral(f, x)
print(F)

### Задача 5. Найти первообразную $\displaystyle\int \frac{dx}{(x^3+1)(x^5+2)}$, без комплексных чисел.

In [ ]:
f = 1/((x^3+1)*(x^5+2))
F = rat_integral(f, x)
print(F)

In [ ]:
# Проверка: производная должна совпадать с f
print('Проверка в точке x=0.5:', (F.diff(x) - f).subs(x=0.5).n())

### Задача 6. График первообразной $\displaystyle\int \frac{x^4\,dx}{x^4 + x + 1}$ на $-10 < x < 10$.

In [ ]:
f = x^4/(x^4 + x + 1)
F = rat_integral(f, x)
print('Первообразная:', F)

In [ ]:
plot(F, (x, -10, 10), ymin=-15, ymax=15)

### Задача 7. Вычислить $\displaystyle\int_1^2 \frac{x^6\,dx}{x^4 - 2}$ с точностью 100 знаков.

In [ ]:
f = x^6/(x^4 - 2)
F = rat_integral(f, x)
val = F.subs(x=2) - F.subs(x=1)
print(val.n(digits=100))

In [ ]:
# Сравнение с numerical_integral
numerical_integral(f, (1, 2))

### Задача 8. Модифицировать `pfdintegral` так, чтобы коэффициенты выражались в радикалах через `radical_expression`.

In [ ]:
def pfdintegral_rad(f, x):
    """Как pfdintegral, но коэффициенты выражены через радикалы."""
    g = (f).numerator()
    h = (f).denominator()
    if AA[x](h).degree() == 1:
        c = SR(g).radical_expression()
        a = SR(h.subs(x=0)).radical_expression()
        return c*ln(abs(x + a))
    else:
        b0 = SR(g.subs(x=0)).radical_expression()
        b1 = SR(diff(g, x)).radical_expression()
        a0 = SR(h.subs(x=0)).radical_expression()
        a1 = SR(diff(h, x).subs(x=0)).radical_expression()
        s = sqrt(-a1^2 + 4*a0)
        return 1/2*b1*log(a1*x + x^2 + a0) - (a1*b1 - 2*b0)*arctan((2*x + a1)/s)/s

In [ ]:
# Тест на примере из лекции
f = 1/(x^3 + 5)
PFD = FractionField(AA[x])(f).partial_fraction_decomposition()
F = sum(pfdintegral_rad(fr, x) for fr in PFD[1])
print(F)

### Задача 9. Выразить $\displaystyle\int_0^1 \frac{dx}{x^4 - 2}$ как символьное выражение с радикалами.

In [ ]:
def rat_integral_rad(f, x):
    PFD = FractionField(AA[x])(f).partial_fraction_decomposition()
    F = poly_integral(PFD[0], x)
    F += sum(pfdintegral_rad(fr, x) for fr in PFD[1])
    return F

In [ ]:
f = 1/(x^4 - 2)
F = rat_integral_rad(f, x)
val = F.subs(x=1) - F.subs(x=0)
print('Символьное выражение:')
print(val.simplify())
print()
print('Числовое значение:', val.n(digits=50))
print('numerical_integral:', numerical_integral(f, (0, 1)))